In [1]:
# ============================================================
# NLP + K-MEANS CLUSTERING PROJECT
# Dataset: 1mb.csv
# ============================================================

# ============================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# ============================================================

# Run this cell once if the libraries are not installed.

# !pip install pandas numpy scikit-learn nltk matplotlib seaborn


# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import warnings

import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


# ============================================================
# CELL 3: DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK data downloaded successfully!")


# ============================================================
# CELL 4: LOAD DATASET
# ============================================================

FILE_NAME = "1mb.csv"

df = pd.read_csv(FILE_NAME)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())


# ============================================================
# CELL 5: BASIC DATASET INFORMATION
# ============================================================

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# CELL 6: REMOVE DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicates removed:", before - after)


# ============================================================
# CELL 7: FIND TEXT COLUMNS
# ============================================================

print("=" * 60)
print("POSSIBLE TEXT COLUMNS")
print("=" * 60)

text_columns = []

for column in df.columns:
    if df[column].dtype == "object":
        text_columns.append(column)
        print(
            f"{column}: "
            f"data type={df[column].dtype}, "
            f"unique values={df[column].nunique()}"
        )


# ============================================================
# CELL 8: SELECT TEXT COLUMN
# ============================================================

# The code first looks for common text-column names.
# If none are found, it automatically chooses the object
# column containing the most text.

common_text_names = [
    "text",
    "review",
    "reviews",
    "description",
    "content",
    "comment",
    "comments",
    "article",
    "news",
    "title",
    "message",
    "body"
]

TEXT_COLUMN = None

# First try common column names
for column in common_text_names:
    if column in df.columns:
        TEXT_COLUMN = column
        break

# If no common name exists, find the object column
# with the largest average text length.
if TEXT_COLUMN is None:

    if len(text_columns) == 0:
        raise ValueError(
            "No text column was found in the CSV file."
        )

    average_lengths = {}

    for column in text_columns:
        average_lengths[column] = (
            df[column]
            .fillna("")
            .astype(str)
            .str.len()
            .mean()
        )

    TEXT_COLUMN = max(
        average_lengths,
        key=average_lengths.get
    )

print("\nSelected text column:")
print(TEXT_COLUMN)


# ============================================================
# CELL 9: PREVIEW TEXT DATA
# ============================================================

print("=" * 60)
print("TEXT DATA PREVIEW")
print("=" * 60)

display(
    df[[TEXT_COLUMN]]
    .head(10)
)


# ============================================================
# CELL 10: CREATE NLP CLEANING FUNCTION
# ============================================================

stop_words = set(
    stopwords.words("english")
)

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep only alphabetic characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 2
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


# ============================================================
# CELL 11: APPLY NLP PREPROCESSING
# ============================================================

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("")

df["clean_text"] = (
    df[TEXT_COLUMN]
    .astype(str)
    .apply(clean_text)
)

print("NLP preprocessing completed!")

display(
    df[[TEXT_COLUMN, "clean_text"]]
    .head(10)
)


# ============================================================
# CELL 12: REMOVE EMPTY TEXT
# ============================================================

before = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Empty rows removed:", before - after)


# ============================================================
# CELL 13: TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print("=" * 60)
print("TF-IDF INFORMATION")
print("=" * 60)

print("Number of documents:", X.shape[0])
print("Number of features:", X.shape[1])
print("Matrix shape:", X.shape)


# ============================================================
# CELL 14: GET TF-IDF FEATURE NAMES
# ============================================================

feature_names = vectorizer.get_feature_names_out()

print("Number of TF-IDF features:", len(feature_names))

print("\nFirst 50 features:")
print(feature_names[:50])


# ============================================================
# CELL 15: FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

# We test K from 2 to 10.

k_values = range(2, 11)

inertia_values = []
silhouette_values = []

print("=" * 60)
print("TESTING DIFFERENT K VALUES")
print("=" * 60)

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
        max_iter=300
    )

    labels = kmeans.fit_predict(X)

    inertia_values.append(
        kmeans.inertia_
    )

    silhouette = silhouette_score(
        X,
        labels
    )

    silhouette_values.append(
        silhouette
    )

    print(
        f"K = {k:2d} | "
        f"Inertia = {kmeans.inertia_:.2f} | "
        f"Silhouette = {silhouette:.4f}"
    )


# ============================================================
# CELL 16: ELBOW METHOD
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    inertia_values,
    marker="o"
)

plt.title(
    "Elbow Method for Choosing K"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 17: SILHOUETTE SCORE
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_values,
    marker="o"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 18: AUTOMATICALLY SELECT K
# ============================================================

best_index = np.argmax(
    silhouette_values
)

BEST_K = list(k_values)[best_index]

print(
    "Best K based on silhouette score:",
    BEST_K
)

print(
    "Best silhouette score:",
    round(
        silhouette_values[best_index],
        4
    )
)


# ============================================================
# CELL 19: RUN FINAL K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10,
    max_iter=300
)

df["cluster"] = kmeans.fit_predict(X)

print("K-Means clustering completed!")


# ============================================================
# CELL 20: CLUSTER DISTRIBUTION
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("=" * 60)
print("CLUSTER DISTRIBUTION")
print("=" * 60)

print(cluster_counts)


# ============================================================
# CELL 21: CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

cluster_counts.plot(
    kind="bar"
)

plt.title(
    "Number of Documents in Each Cluster"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Documents"
)

plt.xticks(
    rotation=0
)

plt.grid(
    axis="y"
)

plt.show()


# ============================================================
# CELL 22: FIND TOP WORDS FOR EACH CLUSTER
# ============================================================

print("=" * 60)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 60)

cluster_keywords = {}

for cluster_number in range(BEST_K):

    center = (
        kmeans
        .cluster_centers_[cluster_number]
    )

    top_indices = (
        center
        .argsort()[-15:][::-1]
    )

    top_words = [
        feature_names[index]
        for index in top_indices
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# CELL 23: CREATE CLUSTER NAMES
# ============================================================

# Automatically create a simple cluster name
# using the top 3 keywords.

cluster_names = {}

for cluster_number, words in cluster_keywords.items():

    cluster_names[
        cluster_number
    ] = (
        " / ".join(words[:3])
    )

print("=" * 60)
print("CLUSTER NAMES")
print("=" * 60)

for cluster, name in cluster_names.items():

    print(
        f"Cluster {cluster}: {name}"
    )


# ============================================================
# CELL 24: ADD CLUSTER NAME TO DATASET
# ============================================================

df["cluster_name"] = (
    df["cluster"]
    .map(cluster_names)
)

display(
    df[
        [
            TEXT_COLUMN,
            "cluster",
            "cluster_name"
        ]
    ].head(20)
)


# ============================================================
# CELL 25: REDUCE DIMENSIONS FOR VISUALIZATION
# ============================================================

# TF-IDF can contain thousands of dimensions.
# TruncatedSVD reduces it to 2 dimensions.

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

print(
    "Reduced matrix shape:",
    X_2d.shape
)


# ============================================================
# CELL 26: VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    cmap="tab10",
    alpha=0.7,
    s=40
)

plt.title(
    "K-Means Clustering of NLP Documents"
)

plt.xlabel(
    "Component 1"
)

plt.ylabel(
    "Component 2"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 27: DISPLAY SAMPLE DOCUMENTS FROM EACH CLUSTER
# ============================================================

print("=" * 70)
print("SAMPLE DOCUMENTS FROM EACH CLUSTER")
print("=" * 70)

for cluster_number in range(BEST_K):

    print("\n")
    print("=" * 70)
    print(
        f"CLUSTER {cluster_number} "
        f"({cluster_names[cluster_number]})"
    )
    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    samples = cluster_data[
        TEXT_COLUMN
    ].head(5)

    for i, text in enumerate(
        samples,
        start=1
    ):

        print(
            f"\n{i}. {text}"
        )


# ============================================================
# CELL 28: CLUSTER SUMMARY TABLE
# ============================================================

summary = []

for cluster_number in range(BEST_K):

    count = (
        df["cluster"]
        .value_counts()
        .get(cluster_number, 0)
    )

    keywords = ", ".join(
        cluster_keywords[
            cluster_number
        ][:10]
    )

    percentage = (
        count / len(df) * 100
    )

    summary.append({
        "Cluster": cluster_number,
        "Documents": count,
        "Percentage": round(
            percentage,
            2
        ),
        "Top Keywords": keywords
    })

cluster_summary = pd.DataFrame(
    summary
)

display(
    cluster_summary
)


# ============================================================
# CELL 29: MOST REPRESENTATIVE DOCUMENTS
# ============================================================

# Distance from each document to its cluster center.
# Smaller distance = more representative document.

distances = kmeans.transform(X)

df["cluster_distance"] = [
    distances[i, cluster]
    for i, cluster
    in enumerate(df["cluster"])
]

print("=" * 70)
print("MOST REPRESENTATIVE DOCUMENTS")
print("=" * 70)

for cluster_number in range(BEST_K):

    representative = (
        df[
            df["cluster"] == cluster_number
        ]
        .sort_values(
            "cluster_distance"
        )
        .head(3)
    )

    print(
        f"\nCLUSTER {cluster_number}"
    )

    for text in representative[
        TEXT_COLUMN
    ]:

        print(
            "\n-",
            text
        )


# ============================================================
# CELL 30: SAVE CLUSTERED DATASET
# ============================================================

OUTPUT_FILE = "1mb_clustered.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"Dataset saved successfully as: {OUTPUT_FILE}"
)


# ============================================================
# CELL 31: SAVE CLUSTER SUMMARY
# ============================================================

SUMMARY_FILE = "cluster_summary.csv"

cluster_summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    f"Cluster summary saved as: {SUMMARY_FILE}"
)


# ============================================================
# CELL 32: FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "\nOriginal dataset:"
)

print(
    f"Rows: {df.shape[0]}"
)

print(
    f"Columns: {df.shape[1]}"
)

print(
    f"\nText column: {TEXT_COLUMN}"
)

print(
    f"\nNumber of clusters: {BEST_K}"
)

print(
    f"Silhouette score: "
    f"{silhouette_values[best_index]:.4f}"
)

print(
    "\nCluster sizes:"
)

print(
    cluster_counts
)

print(
    "\nTop keywords:"
)

for cluster_number in range(BEST_K):

    print(
        f"\nCluster {cluster_number}: "
        f"{cluster_names[cluster_number]}"
    )

print(
    "\nOutput files:"
)

print(
    "- 1mb_clustered.csv"
)

print(
    "- cluster_summary.csv"
)

print("\nProcessing completed successfully!")

Libraries imported successfully!
NLTK data downloaded successfully!
Dataset loaded successfully!
Number of rows: 950
Number of columns: 9


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,first_name,last_name,email,gender,ip_address,date,image,animal,avatar
0,Ancell,Silverson,asilverson0@bbc.co.uk,Male,9.196.147.78,6/6/2017,"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",Guanaco,https://robohash.org/eligendinonaut.bmp?size=5...
1,Waylin,Anscombe,wanscombe1@cnet.com,Male,41.38.11.129,10/22/2017,"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...","Snake, eastern indigo",https://robohash.org/estutneque.jpg?size=50x50...
2,Scotty,Ciric,sciric2@fc2.com,Male,34.160.88.243,6/16/2017,"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...","Rattlesnake, dusky",https://robohash.org/etetmolestias.bmp?size=50...
3,Antoinette,Kovelmann,akovelmann3@harvard.edu,Female,46.158.101.26,5/13/2017,"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",Thomson's gazelle,https://robohash.org/pariaturaliquidamet.bmp?s...
4,Ario,Allsepp,aallsepp4@yelp.com,Male,10.60.67.86,1/30/2017,"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...","Cardinal, red-capped",https://robohash.org/assumendasedaut.bmp?size=...


DATASET INFORMATION

Shape:
(950, 9)

Columns:
['first_name', 'last_name', 'email', 'gender', 'ip_address', 'date', 'image', 'animal', 'avatar']

Data types:
first_name    str
last_name     str
email         str
gender        str
ip_address    str
date          str
image         str
animal        str
avatar        str
dtype: object

Missing values:
first_name    0
last_name     0
email         0
gender        0
ip_address    0
date          0
image         0
animal        0
avatar        0
dtype: int64

Duplicate rows:
0
Rows before removing duplicates: 950
Rows after removing duplicates: 950
Duplicates removed: 0
POSSIBLE TEXT COLUMNS


ValueError: No text column was found in the CSV file.